# Task 2 - Full Training Run with Best Parameters

This notebook is the production-style full run. It is a task-focused version of the
existing end-to-end notebook with the TODO parameters baked in:

- `LIGHT_MODE = False`
- all clean examples, no training cap
- `epochs = 3`
- `LoRA r = 32`, `alpha = 64`
- `lr = 2e-4`, `cosine` schedule
- evaluate on 50 held-out scenarios
- log everything to W&B
- save adapters to Google Drive
- use `dtype=` instead of `torch_dtype=`


## Repo Understanding

This project is a forked AssetOpsBench workflow for training Gemma to plan without tool descriptions.

- `src/servers/` contains the local MCP servers: `iot`, `fmsr`, `tsfm`, and `utilities`.
- `src/workflow/` contains the plan / execute runner used by the benchmark scripts.
- `benchmark/generate_data/datasets/` contains the three SFT datasets used by the current notebook:
  `tool_knowledge.jsonl`, `planning.jsonl`, and `execution.jsonl`.
- `benchmark/baseline_tests/gemini_flash_informed_results.json` is the main gold-plan file used by evaluation.
- `notebook/AssetOpsBench_Gemma_FineTuning.ipynb` is the current end-to-end Colab notebook.

Current notebook gaps that matter for the TODO list:

- training still uses `report_to="none"` in `SFTConfig`
- model loading still uses `torch_dtype=...` instead of `dtype=...`
- pipeline evaluation drops dependencies by hardcoding `Dependency: None`
- the specialist pipeline currently logs no latency analysis, confusion matrix, or W&B tables


## Repo-Specific Audit Notes

The current gold-plan file contains WorkOrder-like scenarios whose gold plans still route
through `IoTAgent`, `none`, or empty plans, while the local training data already uses
`WorkOrderAgent` in many matching scenarios. This notebook includes a normalization cell
so the evaluation can be audited explicitly before the full run is accepted.


In [ ]:
!pip install -U transformers peft trl accelerate bitsandbytes
!pip install -U datasets evaluate rouge-score pandas matplotlib seaborn scikit-learn
!pip install -U wandb sentencepiece protobuf


In [ ]:
import os
import re
import gc
import json
import math
import time
import random
import warnings
import inspect
from copy import deepcopy
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import wandb

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("gpu:", torch.cuda.get_device_name(0), f"({props.total_memory / 1e9:.1f} GB)")


In [ ]:
WANDB_PROJECT = "hpml-group20-assetops"
LIGHT_MODE = False
MODEL_ID = "google/gemma-4-E4B-it"
LOAD_IN_8BIT = True
LOAD_IN_4BIT = False
HF_TOKEN = os.environ.get("HF_TOKEN", "")

REPO_URL = "https://github.com/YuvalShemla/hpml-2026-project.git"
REPO_DIR = Path("/content/hpml-2026-project")
OUTPUT_DIR = Path("/content/output_full_run")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = "all-linear"

GPU_MEM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 40
if GPU_MEM_GB > 60:
    MAX_SEQ_LENGTH = 2048
    PER_DEVICE_BATCH_SIZE = 4
    GRADIENT_ACCUMULATION_STEPS = 4
else:
    MAX_SEQ_LENGTH = 1024
    PER_DEVICE_BATCH_SIZE = 2
    GRADIENT_ACCUMULATION_STEPS = 8

LEARNING_RATE = 2e-4
NUM_EPOCHS = 3
WARMUP_RATIO = 0.05
WEIGHT_DECAY = 0.01
LR_SCHEDULER = "cosine"
BF16 = True
NUM_HELD_OUT = 50
MAX_NEW_TOKENS = 1024
TEMPERATURE = 0.1
TOP_P = 0.9
MAX_TRAIN_EXAMPLES = None

RUN_TAGS = ["full-run", "best-config"]
RUN_GROUP = "task2-full-run"


In [ ]:
if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("repo already present:", REPO_DIR)

def load_json(path):
    with open(path) as f:
        return json.load(f)

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

datasets_dir = REPO_DIR / "benchmark" / "generate_data" / "datasets"
ds_tool = load_jsonl(datasets_dir / "tool_knowledge.jsonl")
ds_plan = load_jsonl(datasets_dir / "planning.jsonl")
ds_exec = load_jsonl(datasets_dir / "execution.jsonl")

gold_path = REPO_DIR / "benchmark" / "baseline_tests" / "gemini_flash_informed_results.json"
gold_plans_raw = load_json(gold_path)
gold_plans = {row["id"]: row for row in gold_plans_raw if row.get("plan_steps", 0) > 0}

print(
    {
        "tool_examples": len(ds_tool),
        "planning_examples": len(ds_plan),
        "execution_examples": len(ds_exec),
        "gold_plans": len(gold_plans),
    }
)


In [ ]:
from datasets import load_dataset

hf_ds = load_dataset("ibm-research/AssetOpsBench", "scenarios")
hf_scenarios = [dict(row) for row in hf_ds["train"]]

eval_candidates = []
for sc in hf_scenarios:
    if sc["id"] in gold_plans:
        eval_candidates.append(
            {
                "id": sc["id"],
                "question": sc["text"],
                "type": sc["type"],
                "category": sc["category"],
                "gold_plan": gold_plans[sc["id"]]["response"],
                "gold_steps": gold_plans[sc["id"]]["plan_steps"],
                "gold_agents": gold_plans[sc["id"]].get("agents_used", []),
                "gold_tools": gold_plans[sc["id"]].get("tools_used", []),
            }
        )

random.shuffle(eval_candidates)
by_type = defaultdict(list)
for sc in eval_candidates:
    by_type[sc["type"]].append(sc)

eval_scenarios = []
per_type_quota = max(1, NUM_HELD_OUT // len(by_type))
for scenario_type, rows in by_type.items():
    eval_scenarios.extend(rows[: min(per_type_quota, len(rows))])

remaining = NUM_HELD_OUT - len(eval_scenarios)
eval_ids = {row["id"] for row in eval_scenarios}
for scenario_type in sorted(by_type, key=lambda key: -len(by_type[key])):
    for row in by_type[scenario_type]:
        if remaining <= 0:
            break
        if row["id"] not in eval_ids:
            eval_scenarios.append(row)
            eval_ids.add(row["id"])
            remaining -= 1

eval_questions_lower = {row["question"].strip().lower() for row in eval_scenarios}
held_out_plans = {row["gold_plan"].strip() for row in eval_scenarios}

def is_leaked(example):
    return example["messages"][0]["content"].strip().lower() in eval_questions_lower

clean_tool = [row for row in ds_tool if not is_leaked(row)]
clean_plan = [row for row in ds_plan if not is_leaked(row)]
clean_exec = [row for row in ds_exec if not is_leaked(row)]
clean_plan = [row for row in clean_plan if row["messages"][1]["content"].strip() not in held_out_plans]

print("held_out:", len(eval_scenarios))
print("type_distribution:", dict(Counter(row["type"] for row in eval_scenarios)))
print("clean_train_total:", len(clean_tool) + len(clean_plan) + len(clean_exec))


In [ ]:
from evaluate import load as load_metric

rouge_metric = load_metric("rouge")

VALID_AGENTS = {
    "IoTAgent",
    "FMSRAgent",
    "TSFMAgent",
    "Utilities",
    "WorkOrderAgent",
    "VibrationAgent",
    "none",
}

VALID_TOOLS = {
    "sites",
    "assets",
    "sensors",
    "history",
    "get_failure_modes",
    "get_failure_mode_sensor_mapping",
    "get_ai_tasks",
    "get_tsfm_models",
    "run_tsfm_forecasting",
    "run_tsfm_finetuning",
    "run_tsad",
    "run_integrated_tsad",
    "json_reader",
    "current_date_time",
    "current_time_english",
    "get_work_orders",
    "get_preventive_work_orders",
    "get_corrective_work_orders",
    "get_events",
    "get_failure_codes",
    "get_work_order_distribution",
    "predict_next_work_order",
    "analyze_alert_to_failure",
    "get_vibration_data",
    "list_vibration_sensors",
    "compute_fft_spectrum",
    "compute_envelope_spectrum",
    "assess_vibration_severity",
    "calculate_bearing_frequencies",
    "list_known_bearings",
    "diagnose_vibration",
    "none",
}

TOOL_DESCRIPTIONS = """Available Agents and Tools:

IoTAgent:
  - sites()
  - assets(site_name)
  - sensors(site_name, asset_id)
  - history(site_name, asset_id, start, final=None)

FMSRAgent:
  - get_failure_modes(asset_name)
  - get_failure_mode_sensor_mapping(asset_name, failure_modes, sensors)

TSFMAgent:
  - get_ai_tasks()
  - get_tsfm_models()
  - run_tsfm_forecasting(...)
  - run_tsfm_finetuning(...)
  - run_tsad(...)
  - run_integrated_tsad(...)

Utilities:
  - json_reader(file_name)
  - current_date_time()
  - current_time_english()

WorkOrderAgent:
  - get_work_orders(asset_name)
  - get_preventive_work_orders(asset_name)
  - get_corrective_work_orders(asset_name)
  - get_events(asset_name)
  - get_failure_codes(asset_name)
  - get_work_order_distribution(asset_name)
  - predict_next_work_order(asset_name)
  - analyze_alert_to_failure(asset_name)
"""

INFORMED_PROMPT = """You are an expert planner for industrial asset operations. Given a question and available tools, produce a structured plan.

{tool_descriptions}

OUTPUT FORMAT:
#Task1: <description>
#Agent1: <agent_name>
#Tool1: <tool_name>
#Args1: {{"arg": "value"}}
#Dependency1: None
#ExpectedOutput1: <what to expect>

Rules: Use only listed agents/tools. Keep plans concise. Use {{step_N}} for dependencies.

QUESTION: {question}
"""

BLIND_PROMPT = """You are an expert planner for industrial asset operations. Produce a structured plan.

OUTPUT FORMAT:
#Task1: <description>
#Agent1: <agent_name>
#Tool1: <tool_name>
#Args1: {{"arg": "value"}}
#Dependency1: None
#ExpectedOutput1: <what to expect>

Rules: Keep plans concise. Use {{step_N}} for dependencies.

QUESTION: {question}
"""

def parse_plan(text):
    steps = []
    if not text:
        return steps
    for block in re.split(r"(?=#Task\d+:)", text):
        block = block.strip()
        task_m = re.search(r"#Task(\d+):\s*(.*)", block)
        agent_m = re.search(r"#Agent\d+:\s*(\S+)", block)
        tool_m = re.search(r"#Tool\d+:\s*(\S+)", block)
        args_m = re.search(r"#Args\d+:\s*(.*)", block)
        dep_m = re.search(r"#Dependency\d+:\s*(.*)", block)
        exp_m = re.search(r"#ExpectedOutput\d+:\s*(.*)", block)
        if not task_m:
            continue
        args_raw = args_m.group(1).strip() if args_m else "{}"
        try:
            args = json.loads(args_raw)
        except Exception:
            args = {}
        steps.append(
            {
                "step": int(task_m.group(1)),
                "task": task_m.group(2).strip(),
                "agent": agent_m.group(1).strip() if agent_m else "",
                "tool": tool_m.group(1).strip().rstrip("()") if tool_m else "",
                "args": args,
                "args_raw": args_raw,
                "dependency": dep_m.group(1).strip() if dep_m else "None",
                "expected_output": exp_m.group(1).strip() if exp_m else "",
            }
        )
    return steps

def compute_set_f1(gold_items, pred_items):
    gold_set = set(gold_items)
    pred_set = set(pred_items)
    if not gold_set and not pred_set:
        return 1.0
    if not gold_set or not pred_set:
        return 0.0
    tp = len(gold_set & pred_set)
    precision = tp / len(pred_set)
    recall = tp / len(gold_set)
    return 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

def dependency_edge_set(steps):
    edges = set()
    for step in steps:
        dep_field = step.get("dependency", "")
        dep_ids = re.findall(r"(?:step_|#S)?(\d+)", dep_field)
        for dep_id in dep_ids:
            edges.add((int(dep_id), step["step"]))
    return edges

def evaluate_plan(pred_text, gold_text):
    pred_steps = parse_plan(pred_text)
    gold_steps = parse_plan(gold_text)

    pred_pairs = [(step["agent"], step["tool"]) for step in pred_steps if step["tool"]]
    gold_pairs = [(step["agent"], step["tool"]) for step in gold_steps if step["tool"]]

    gold_args = {(step["agent"], step["tool"]): step["args"] for step in gold_steps if step["args"]}

    key_scores = []
    value_scores = []
    for step in pred_steps:
        key = (step["agent"], step["tool"])
        if key not in gold_args:
            continue
        gold_arg_dict = gold_args[key]
        pred_arg_dict = step["args"] if isinstance(step["args"], dict) else {}
        key_scores.append(compute_set_f1(gold_arg_dict.keys(), pred_arg_dict.keys()))
        shared = set(gold_arg_dict) & set(pred_arg_dict)
        if shared:
            value_scores.append(
                np.mean(
                    [
                        1.0
                        if str(gold_arg_dict[k]).strip().lower() == str(pred_arg_dict[k]).strip().lower()
                        else 0.0
                        for k in shared
                    ]
                )
            )

    dep_f1 = compute_set_f1(dependency_edge_set(gold_steps), dependency_edge_set(pred_steps))
    rouge = rouge_metric.compute(predictions=[pred_text], references=[gold_text])["rougeL"]

    return {
        "has_plan_format": "#Task" in pred_text and "#Agent" in pred_text and "#Tool" in pred_text,
        "num_steps": len(pred_steps),
        "gold_steps": len(gold_steps),
        "agent_tool_f1": compute_set_f1(gold_pairs, pred_pairs),
        "arg_key_f1": float(np.mean(key_scores)) if key_scores else 0.0,
        "arg_value_match": float(np.mean(value_scores)) if value_scores else 0.0,
        "dependency_f1": dep_f1,
        "rouge_l": rouge,
        "valid_agents": sum(1 for step in pred_steps if step["agent"] in VALID_AGENTS),
        "total_agents": len(pred_steps),
        "valid_tools": sum(1 for step in pred_steps if step["tool"] in VALID_TOOLS),
        "total_tools": len(pred_steps),
        "pred_steps": pred_steps,
        "gold_steps_parsed": gold_steps,
    }

def summarize_results(results, mode_name):
    if not results:
        return {"mode": mode_name, "total": 0}
    return {
        "mode": mode_name,
        "total": len(results),
        "format_valid_pct": 100 * np.mean([row["has_plan_format"] for row in results]),
        "avg_agent_tool_f1": float(np.mean([row["agent_tool_f1"] for row in results])),
        "avg_arg_key_f1": float(np.mean([row["arg_key_f1"] for row in results])),
        "avg_arg_value_match": float(np.mean([row["arg_value_match"] for row in results])),
        "avg_dependency_f1": float(np.mean([row["dependency_f1"] for row in results])),
        "avg_rouge_l": float(np.mean([row["rouge_l"] for row in results])),
        "agent_correctness": sum(row["valid_agents"] for row in results) / max(sum(row["total_agents"] for row in results), 1),
        "tool_correctness": sum(row["valid_tools"] for row in results) / max(sum(row["total_tools"] for row in results), 1),
    }

def print_summary(summary):
    print(json.dumps(summary, indent=2))


In [ ]:
def build_run_config(extra=None):
    config = {
        "model": MODEL_ID,
        "light_mode": LIGHT_MODE,
        "load_in_8bit": LOAD_IN_8BIT,
        "load_in_4bit": LOAD_IN_4BIT,
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        "learning_rate": LEARNING_RATE,
        "epochs": NUM_EPOCHS,
        "per_device_batch_size": PER_DEVICE_BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "max_seq_length": MAX_SEQ_LENGTH,
        "held_out_scenarios": NUM_HELD_OUT,
        "dataset_sizes": {
            "tool": len(ds_tool),
            "planning": len(ds_plan),
            "execution": len(ds_exec),
        },
        "clean_dataset_sizes": {
            "tool": len(clean_tool),
            "planning": len(clean_plan),
            "execution": len(clean_exec),
        },
    }
    if extra:
        config.update(extra)
    return config

def start_run(name, tags=None, config=None, group=None):
    return wandb.init(
        project=WANDB_PROJECT,
        name=name,
        tags=tags or [],
        group=group or RUN_GROUP,
        config=build_run_config(config),
        reinit=True,
    )

def log_trainer_history(trainer, prefix):
    history = trainer.state.log_history
    for row in history:
        payload = {}
        if "step" in row:
            payload["step"] = row["step"]
        for key in ("loss", "eval_loss", "learning_rate", "epoch"):
            if key in row:
                payload[f"{prefix}/{key}"] = row[key]
        if payload:
            wandb.log(payload)

def dataframe_table(df, name):
    if df is None or df.empty:
        return None
    table = wandb.Table(dataframe=df.reset_index(drop=True))
    wandb.log({name: table})
    return table

def results_to_dataframe(results):
    rows = []
    for row in results:
        rows.append(
            {
                "id": row["id"],
                "type": row.get("type", ""),
                "mode": row.get("mode", ""),
                "agent_tool_f1": row.get("agent_tool_f1", 0.0),
                "arg_key_f1": row.get("arg_key_f1", 0.0),
                "arg_value_match": row.get("arg_value_match", 0.0),
                "dependency_f1": row.get("dependency_f1", 0.0),
                "rouge_l": row.get("rouge_l", 0.0),
                "input_tokens": row.get("input_tokens", 0),
                "output_tokens": row.get("output_tokens", 0),
                "question": row.get("question", ""),
                "generated": row.get("generated", "")[:800],
            }
        )
    return pd.DataFrame(rows)

def log_summary(summary, prefix):
    wandb.log({f"{prefix}/{key}": value for key, value in summary.items() if isinstance(value, (int, float, bool))})

def finish_run():
    if wandb.run is not None:
        wandb.finish()


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

if LOAD_IN_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    quant_label = "4-bit NF4"
else:
    bnb_config = BitsAndBytesConfig(load_in_8bit=True)
    quant_label = "8-bit"

print("loading", MODEL_ID, "with", quant_label)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.bfloat16,
    token=HF_TOKEN,
    attn_implementation="eager",
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id


In [ ]:
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

def setup_lora(base_model):
    peft_model = prepare_model_for_kbit_training(base_model)
    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGET_MODULES,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )
    peft_model = get_peft_model(peft_model, lora_config)
    trainable, total = peft_model.get_nb_trainable_parameters()
    print(f"trainable: {trainable:,} / {total:,}")
    return peft_model

def train_model(peft_model, train_data, eval_data, output_dir, run_name, report_to="wandb", tags=None):
    os.makedirs(output_dir, exist_ok=True)
    train_ds = Dataset.from_list(train_data)
    eval_ds = Dataset.from_list(eval_data) if eval_data else None

    total_steps = max(1, len(train_ds) * NUM_EPOCHS // max(PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS, 1))
    warmup_steps = max(1, int(total_steps * WARMUP_RATIO))

    sft_kwargs = dict(
        output_dir=output_dir,
        run_name=run_name,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type=LR_SCHEDULER,
        warmup_steps=warmup_steps,
        weight_decay=WEIGHT_DECAY,
        bf16=BF16,
        logging_steps=10,
        eval_strategy="steps" if eval_ds else "no",
        eval_steps=50 if eval_ds else None,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=2,
        load_best_model_at_end=bool(eval_ds),
        report_to=report_to,
        seed=SEED,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        remove_unused_columns=False,
    )
    if "max_seq_length" in inspect.signature(SFTConfig.__init__).parameters:
        sft_kwargs["max_seq_length"] = MAX_SEQ_LENGTH
    sft_config = SFTConfig(**sft_kwargs)

    trainer_kwargs = {
        "model": peft_model,
        "args": sft_config,
        "train_dataset": train_ds,
        "processing_class": tokenizer,
    }
    if eval_ds:
        trainer_kwargs["eval_dataset"] = eval_ds
    if (
        "max_seq_length" in inspect.signature(SFTTrainer.__init__).parameters
        and "max_seq_length" not in sft_kwargs
    ):
        trainer_kwargs["max_seq_length"] = MAX_SEQ_LENGTH

    trainer = SFTTrainer(**trainer_kwargs)
    trainer.train()
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    return trainer


In [ ]:
def strip_plan_to_routing(plan_text):
    keep = []
    for line in plan_text.splitlines():
        stripped = line.strip()
        if not stripped:
            keep.append("")
            continue
        if stripped.startswith(("#Task", "#Agent", "#Tool", "#Dependency")):
            keep.append(line)
    return "\n".join(keep).strip()

def extract_specialist_steps(plan_text):
    steps = []
    for step in parse_plan(plan_text):
        if step["agent"] in ("", "none"):
            continue
        instruction = (
            "Generate the arguments for this tool call:\n"
            f"Task: {step['task']}\n"
            f"Agent: {step['agent']}\n"
            f"Tool: {step['tool']}\n"
            f"Dependency: {step['dependency']}"
        )
        response = f"#Args: {json.dumps(step['args'])}\n#ExpectedOutput: {step['expected_output'] or 'Result'}"
        steps.append({"agent": step["agent"], "instruction": instruction, "response": response})
    return steps

def build_training_sets():
    all_train = [{"messages": row["messages"]} for row in clean_tool + clean_plan + clean_exec]
    random.shuffle(all_train)
    if MAX_TRAIN_EXAMPLES:
        all_train = all_train[:MAX_TRAIN_EXAMPLES]
    split = int(len(all_train) * 0.95)
    generalist_train, generalist_eval = all_train[:split], all_train[split:]

    planner_data = []
    for row in clean_plan:
        if row.get("metadata", {}).get("category") != "planning":
            continue
        routing = strip_plan_to_routing(row["messages"][1]["content"])
        if "#Task" in routing and "#Agent" in routing:
            planner_data.append(
                {
                    "messages": [
                        {"role": "user", "content": row["messages"][0]["content"]},
                        {"role": "assistant", "content": routing},
                    ]
                }
            )
    for row in clean_tool:
        planner_data.append({"messages": row["messages"]})
    random.shuffle(planner_data)
    if MAX_TRAIN_EXAMPLES:
        planner_data = planner_data[:MAX_TRAIN_EXAMPLES]
    split = int(len(planner_data) * 0.95)
    planner_train, planner_eval = planner_data[:split], planner_data[split:]

    specialist_data = defaultdict(list)
    for row in clean_plan:
        if row.get("metadata", {}).get("category") != "planning":
            continue
        for step in extract_specialist_steps(row["messages"][1]["content"]):
            specialist_data[step["agent"]].append(
                {
                    "messages": [
                        {"role": "user", "content": step["instruction"]},
                        {"role": "assistant", "content": step["response"]},
                    ]
                }
            )

    return generalist_train, generalist_eval, planner_train, planner_eval, specialist_data


In [ ]:
workorder_tools = {
    "get_work_orders",
    "get_preventive_work_orders",
    "get_corrective_work_orders",
    "get_events",
    "get_failure_codes",
    "get_work_order_distribution",
    "predict_next_work_order",
    "analyze_alert_to_failure",
}

local_plan_lookup = {}
for row in ds_plan:
    local_plan_lookup[row["messages"][0]["content"].strip().lower()] = row["messages"][1]["content"]

def looks_like_workorder_mismatch(scenario):
    gold_steps = parse_plan(scenario["gold_plan"])
    if not gold_steps:
        return "work order" in scenario["question"].lower()
    tools = {step["tool"] for step in gold_steps}
    agents = {step["agent"] for step in gold_steps}
    if tools & workorder_tools:
        return False
    if "work order" not in scenario["question"].lower():
        return False
    return not ("WorkOrderAgent" in agents)

mismatches = [scenario for scenario in eval_scenarios if looks_like_workorder_mismatch(scenario)]
print("potential_workorder_gold_mismatches:", len(mismatches))
audit_rows = []
for scenario in mismatches[:15]:
    fallback = local_plan_lookup.get(scenario["question"].strip().lower(), "")
    audit_rows.append(
        {
            "id": scenario["id"],
            "type": scenario["type"],
            "question": scenario["question"][:140],
            "gold_agents": [step["agent"] for step in parse_plan(scenario["gold_plan"])],
            "fallback_agents": [step["agent"] for step in parse_plan(fallback)],
        }
    )
pd.DataFrame(audit_rows)


In [ ]:
def normalize_gold_plan(scenario):
    fallback = local_plan_lookup.get(scenario["question"].strip().lower())
    if fallback and looks_like_workorder_mismatch(scenario):
        normalized = dict(scenario)
        normalized["gold_plan"] = fallback
        return normalized
    return scenario

normalized_eval_scenarios = [normalize_gold_plan(scenario) for scenario in eval_scenarios]


In [ ]:
from google.colab import drive

DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/hpml_group20_adapters"
drive.mount("/content/drive", force_remount=True)
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
print("drive output:", DRIVE_OUTPUT_DIR)


In [ ]:
generalist_train, generalist_eval, planner_train, planner_eval, specialist_data = build_training_sets()
TARGET_SPECIALISTS = ["IoTAgent", "FMSRAgent", "TSFMAgent", "WorkOrderAgent"]

baseline_run = start_run(
    name="task2-baseline-full",
    tags=RUN_TAGS + ["baseline"],
    config={"stage": "baseline"},
    group=RUN_GROUP,
)
baseline_blind_results, baseline_blind_summary = run_evaluation(
    model,
    tokenizer,
    normalized_eval_scenarios,
    BLIND_PROMPT,
    "Baseline: Blind",
)
log_summary(baseline_blind_summary, "baseline_blind")
dataframe_table(results_to_dataframe(baseline_blind_results), "baseline_blind/per_scenario")
finish_run()

generalist_run = start_run(
    name="task2-generalist-full",
    tags=RUN_TAGS + ["generalist"],
    config={"stage": "generalist"},
    group=RUN_GROUP,
)
peft_generalist = setup_lora(model)
gen_trainer = train_model(
    peft_generalist,
    generalist_train,
    generalist_eval,
    str(OUTPUT_DIR / "generalist"),
    run_name="task2-generalist",
    report_to="wandb",
)
log_trainer_history(gen_trainer, "generalist")
gen_blind_results, gen_blind_summary = run_evaluation(
    peft_generalist,
    tokenizer,
    normalized_eval_scenarios,
    BLIND_PROMPT,
    "Generalist: Blind",
)
log_summary(gen_blind_summary, "generalist_blind")
dataframe_table(results_to_dataframe(gen_blind_results), "generalist/per_scenario")
finish_run()


In [ ]:
planner_run = start_run(
    name="task2-planner-full",
    tags=RUN_TAGS + ["planner"],
    config={"stage": "planner"},
    group=RUN_GROUP,
)
peft_planner = setup_lora(model)
planner_trainer = train_model(
    peft_planner,
    planner_train,
    planner_eval,
    str(OUTPUT_DIR / "planner"),
    run_name="task2-planner",
    report_to="wandb",
)
log_trainer_history(planner_trainer, "planner")
planner_results, planner_summary = run_evaluation(
    peft_planner,
    tokenizer,
    normalized_eval_scenarios,
    BLIND_PROMPT,
    "Planner: Blind",
)
log_summary(planner_summary, "planner_blind")
dataframe_table(results_to_dataframe(planner_results), "planner/per_scenario")
finish_run()


In [ ]:
specialist_models = {}
specialist_summaries = []

for agent_name in TARGET_SPECIALISTS:
    agent_rows = specialist_data.get(agent_name, [])
    if len(agent_rows) < 20:
        print("skipping specialist", agent_name, "because dataset is too small:", len(agent_rows))
        continue
    agent_run = start_run(
        name=f"task2-specialist-{agent_name}",
        tags=RUN_TAGS + ["specialist", agent_name],
        config={"stage": "specialist", "agent_name": agent_name, "examples": len(agent_rows)},
        group=RUN_GROUP,
    )
    random.shuffle(agent_rows)
    split = int(len(agent_rows) * 0.9)
    spec_model = setup_lora(model)
    trainer = train_model(
        spec_model,
        agent_rows[:split],
        agent_rows[split:] if split < len(agent_rows) else None,
        str(OUTPUT_DIR / f"specialist_{agent_name}"),
        run_name=f"task2-specialist-{agent_name}",
        report_to="wandb",
    )
    log_trainer_history(trainer, f"specialist/{agent_name}")
    specialist_models[agent_name] = spec_model
    specialist_summaries.append({"agent_name": agent_name, "examples": len(agent_rows)})
    finish_run()

pd.DataFrame(specialist_summaries)


In [ ]:
def fill_pipeline_plan(planner_text, specialist_models):
    full_lines = []
    for step in parse_plan(planner_text):
        agent = step["agent"]
        tool = step["tool"]
        dependency = step["dependency"]
        expected_output = ""
        args_text = "{}"

        if agent in specialist_models:
            spec_model = specialist_models[agent]
            spec_prompt = (
                "Generate the arguments for this tool call:\n"
                f"Task: {step['task']}\n"
                f"Agent: {agent}\n"
                f"Tool: {tool}\n"
                f"Dependency: {dependency}"
            )
            chat = [{"role": "user", "content": spec_prompt}]
            tokenized = tokenizer.apply_chat_template(
                chat,
                return_tensors="pt",
                add_generation_prompt=True,
                return_dict=True,
            )
            input_ids = tokenized["input_ids"].to(spec_model.device)
            attention_mask = tokenized["attention_mask"].to(spec_model.device)
            with torch.no_grad():
                output_ids = spec_model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    max_new_tokens=256,
                    temperature=TEMPERATURE,
                    top_p=TOP_P,
                    do_sample=True,
                    pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
                )
            decoded = tokenizer.decode(output_ids[0][input_ids.shape[1]:], skip_special_tokens=True)
            args_match = re.search(r"#Args:\s*(.*)", decoded)
            exp_match = re.search(r"#ExpectedOutput:\s*(.*)", decoded)
            args_text = args_match.group(1).strip() if args_match else "{}"
            expected_output = exp_match.group(1).strip() if exp_match else ""

        full_lines.extend(
            [
                f"#Task{step['step']}: {step['task']}",
                f"#Agent{step['step']}: {agent}",
                f"#Tool{step['step']}: {tool}",
                f"#Args{step['step']}: {args_text}",
                f"#Dependency{step['step']}: {dependency}",
                f"#ExpectedOutput{step['step']}: {expected_output or 'Result'}",
                "",
            ]
        )
    return "\n".join(full_lines).strip()

pipeline_results = []
planner_by_id = {row["id"]: row for row in planner_results}
for scenario in tqdm(normalized_eval_scenarios, desc="pipeline_eval"):
    planner_row = planner_by_id.get(scenario["id"])
    if not planner_row:
        continue
    combined_plan = fill_pipeline_plan(planner_row["generated"], specialist_models)
    metrics = evaluate_plan(combined_plan, scenario["gold_plan"])
    metrics.update(
        {
            "id": scenario["id"],
            "question": scenario["question"],
            "type": scenario["type"],
            "generated": combined_plan,
            "mode": "Pipeline: Planner+Specialists",
        }
    )
    pipeline_results.append(metrics)

pipeline_summary = summarize_results(pipeline_results, "Pipeline: Planner+Specialists")

final_run = start_run(
    name="task2-pipeline-summary",
    tags=RUN_TAGS + ["pipeline", "comparison"],
    config={"stage": "pipeline"},
    group=RUN_GROUP,
)
log_summary(pipeline_summary, "pipeline")
dataframe_table(results_to_dataframe(pipeline_results), "pipeline/per_scenario")

comparison_df = pd.DataFrame(
    [
        {"mode": "Baseline: Blind", **baseline_blind_summary},
        {"mode": "Generalist: Blind", **gen_blind_summary},
        {"mode": "Planner: Blind", **planner_summary},
        {"mode": "Pipeline: Planner+Specialists", **pipeline_summary},
    ]
)
dataframe_table(comparison_df, "comparison/summary")
finish_run()

comparison_df


In [ ]:
for path in OUTPUT_DIR.glob("*"):
    if path.is_dir():
        target = os.path.join(DRIVE_OUTPUT_DIR, path.name)
        !cp -r "{path}" "{target}"
print("copied adapters and outputs to:", DRIVE_OUTPUT_DIR)
